In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

cat_in_the_dat_path = kagglehub.competition_download('cat-in-the-dat')

print('Data source import complete.')


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_path = '/kaggle/input/cat-in-the-dat/'

In [ ]:
train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')
submission = pd.read_csv(data_path + 'sample_submission.csv', index_col='id')

# 피처 엔지니어링

## 데이터 합치기

In [ ]:
all_data = pd.concat([train, test])
all_data = all_data.drop('target', axis=1)
all_data

## 원-핫인코딩

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
encoder = OneHotEncoder()
all_data_encoded = encoder.fit_transform(all_data)

## 데이터 나누기

In [ ]:
num_train = len(train)

In [ ]:
X_train = all_data_encoded[:num_train]
X_test = all_data_encoded[num_train:]

In [ ]:
y = train['target']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y,
                                                      test_size=0.1,
                                                      stratify=y,
                                                      random_state=10)

# 모델 훈련

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train, y_train)

In [ ]:
logistic_model.predict_proba(X_valid)

In [ ]:
logistic_model.predict(X_valid)

In [ ]:
y_valid_preds = logistic_model.predict_proba(X_valid)[:, 1] # 타깃값이 1일 확률을 저장

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
roc_auc = roc_auc_score(y_valid, y_valid_preds)

print(f'검증 데이터 ROC AUC: {roc_auc:.4f}')

# 예측 및 결과 제출

In [ ]:
y_preds = logistic_model.predict_proba(X_test)[:, 1]

In [ ]:
submission['target'] = y_preds
submission.to_csv('submission.csv')

# 성능 개선 1

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_path = '/kaggle/input/cat-in-the-dat/'

In [ ]:
train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')
submission = pd.read_csv(data_path + 'sample_submission.csv', index_col='id')

## 피처 엔지니어링 1: 피처 맞춤 인코딩

### 데이터 합치기

In [ ]:
all_data = pd.concat([train, test])
all_data = all_data.drop('target', axis=1)

### 이진 피처 인코딩

In [ ]:
all_data['bin_3'] = all_data['bin_3'].map({'F':0, 'T':1})
all_data['bin_4'] = all_data['bin_4'].map({'N':0, 'Y':1})

### 순서 피처 인코딩

In [ ]:
ord1dict = {'Novice':0, 'Contributor':1, 'Expert':2, 'Master':3, 'Grandmaster':4}
ord2dict = {'Freezing':0, 'Cold':1, 'Warm':2, 'Hot':3, 'Boiling Hot':4, 'Lava Hot':5}

In [ ]:
all_data['ord_1'] = all_data['ord_1'].map(ord1dict)
all_data['ord_2'] = all_data['ord_2'].map(ord2dict)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
ord_345 = ['ord_3', 'ord_4', 'ord_5']

In [ ]:
ord_encoder = OrdinalEncoder()

In [ ]:
all_data[ord_345] = ord_encoder.fit_transform(all_data[ord_345])

In [ ]:
for feature, categories in zip(ord_345, ord_encoder.categories_):
    print(feature)
    print(categories)

### 명목형 피처 인코딩

In [ ]:
nom_features = ['nom_' + str(i) for i in range(10)]

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
onehot_encoder = OneHotEncoder()

In [ ]:
encoded_nom_matrix = onehot_encoder.fit_transform(all_data[nom_features])
encoded_nom_matrix # COO와 CSR 형식 중 CSR 형식

In [ ]:
all_data = all_data.drop(nom_features, axis=1) # 기존 명목형 피처 삭제

### 날짜 피처 인코딩

In [ ]:
date_features = ['day', 'month']

In [ ]:
encoded_date_matirx = onehot_encoder.fit_transform(all_data[date_features])

In [ ]:
all_data = all_data.drop(date_features, axis=1)

In [ ]:
encoded_date_matirx

## 피처 엔지니어링 2: 피처 스케일링

### 순서형 피처 스케일링

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
ord_features = ['ord_' + str(i) for i in range(6)]

In [ ]:
all_data[ord_features] = MinMaxScaler().fit_transform(all_data[ord_features])

### 인코딩 및 스케일링된 피처 합치기

In [ ]:
from scipy import sparse

In [ ]:
all_data_sprs = sparse.hstack([sparse.csr_matrix(all_data),
                               encoded_nom_matrix,
                               encoded_date_matirx],
                              format='csr')

In [ ]:
all_data_sprs

### 데이터 나누기

In [ ]:
num_train = len(train)

In [ ]:
X_train = all_data_sprs[:num_train]
X_test = all_data_sprs[num_train:]

In [ ]:
y = train['target']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y,
                                                      test_size=0.1,
                                                      stratify=y,
                                                      random_state=10)

## 하이퍼파라미터 최적화

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

In [ ]:
%%time

logistic_model = LogisticRegression()

lr_params = {'C':[0.1, 0.125, 0.2], 'max_iter':[800, 900, 1000],
             'solver':['liblinear'], 'random_state':[42]}

gridsearch_logistic_model = GridSearchCV(estimator=logistic_model,
                                         param_grid=lr_params,
                                         scoring='roc_auc',
                                         cv=5)

gridsearch_logistic_model.fit(X_train, y_train)

print('최적 하이퍼파라미터: ', gridsearch_logistic_model.best_params_)

## 모델 성능 검증

In [ ]:
y_valid_preds = gridsearch_logistic_model.predict_proba(X_valid)[:,1]

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
roc_auc = roc_auc_score(y_valid, y_valid_preds)

In [ ]:
print(f'검증 데이터 ROC AUC: {roc_auc:.4f}')

## 예측 및 결과 제출

In [ ]:
y_preds = gridsearch_logistic_model.best_estimator_.predict_proba(X_test)[:,1]

In [ ]:
submission['target'] = y_preds
submission.to_csv('submission.csv')

# 성능 개선 2

훈련 데이터에서 훈련 데이터와 검증 데이터를 나누지 않고 전체 훈련 데이터로 학습 시키기

In [ ]:
num_train = len(train)

In [ ]:
X_train = all_data_sprs[:num_train]
X_test = all_data_sprs[num_train:]

In [ ]:
y = train['target']

In [ ]:
%%time

logistic_model = LogisticRegression()

lr_params = {'C':[0.1, 0.125, 0.2], 'max_iter':[800, 900, 1000],
             'solver':['liblinear'], 'random_state':[42]}

gridsearch_logistic_model = GridSearchCV(estimator=logistic_model,
                                         param_grid=lr_params,
                                         scoring='roc_auc',
                                         cv=5)

gridsearch_logistic_model.fit(X_train, y)

print('최적 하이퍼파라미터: ', gridsearch_logistic_model.best_params_)

In [ ]:
y_preds = gridsearch_logistic_model.best_estimator_.predict_proba(X_test)[:,1]

In [ ]:
submission['target'] = y_preds
submission.to_csv('submission.csv')